# CRISP-DM Phase 3: Data Preparation (Medical Dataset)

In this phase, we apply transformations based on our findings from Phase 2 (Data Understanding). Our goals are:
1. **Encode the Target:** Turn the string predictions into binary numbers.
2. **Feature Engineering:** Create clinically relevant new variables to help the models.
3. **Preserve Signal Outliers:** Noticeably, heart attacks show up as massive spikes in the Cardiac Biomarkers. We will retain these outliers.
4. **Scale:** Because of the heavy outliers, we will use `RobustScaler` instead of StandardScaler so the models don't get heavily skewed.
5. **Export:** Save cleaned train/test data to the `data/processed` directory.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
import warnings

warnings.filterwarnings('ignore')

## 1. Data Loading
Let's load the raw data used in the previous phase.

In [2]:
DATA_FILE = 'Medicaldataset.csv'
df = pd.read_csv(DATA_FILE)
print(f"Raw Data Loaded. Dimensions: {df.shape}")

Raw Data Loaded. Dimensions: (1319, 9)


## 2. Feature Engineering
Clinical domain logic allows us to create powerful composite features:
- **Pulse_Pressure**: The difference between systolic and diastolic blood pressure.
- **CK_MB_Troponin_Ratio**: The ratio between the two cardiac biomarkers.

In [3]:
df_engineered = df.copy()

# 1. Pulse Pressure
df_engineered['Pulse_Pressure'] = df_engineered['Systolic blood pressure'] - df_engineered['Diastolic blood pressure']

# 2. Biomarker Ratio (protect against divide by zero by adding a tiny epsilon)
epsilon = 1e-9
df_engineered['CK_MB_Troponin_Ratio'] = df_engineered['CK-MB'] / (df_engineered['Troponin'] + epsilon)

print("Features Engineered Successfully.")
display(df_engineered[['Pulse_Pressure', 'CK_MB_Troponin_Ratio']].head())

Features Engineered Successfully.


,Pulse_Pressure,CK_MB_Troponin_Ratio
0,77,149.999988
1,52,6.367925
2,83,663.333112
3,65,113.688524
4,47,359.999880


## 3. Target Encoding
Machine learning algorithms require numerical targets. We convert: 
- `positive` -> 1
- `negative` -> 0

In [4]:
le = LabelEncoder()
df_engineered['Result'] = le.fit_transform(df_engineered['Result'])

# Verify the mapping (positive should typically be 1, negative 0. LabelEncoder sorts alphabetically so 'negative'=0, 'positive'=1)
print("Encoding mapping:")
for idx, class_name in enumerate(le.classes_):
    print(f"{class_name}: {idx}")

Encoding mapping:
negative: 0
positive: 1


## 4. Train/Test Split
Before we scale, we must split our data. Scaling *before* splitting is a cardinal sin called **Data Leakage** (where information from the test set leaks into the training mean/median).
We use an 80/20 split and `stratify=y` to ensure the 61/38% class imbalance is preserved identically in both sets.

In [5]:
X = df_engineered.drop('Result', axis=1)
y = df_engineered['Result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (1055, 10)
Testing set shape: (264, 10)


## 5. Robust Scaling
As discovered in the EDA phase, `Troponin` and `CK-MB` have severe outliers that actually correspond directly to heart attacks. `StandardScaler` calculates the mean of a column, which can be thrown completely off by severe outliers. 

Instead, we use `RobustScaler` which uses the Median and the Interquartile Range, completely ignoring the wild extreme values when calculating the scale.

In [6]:
scaler = RobustScaler()

# We fit ONLY on the training data to prevent leakage, then transform both
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Data scaled robustly.")
display(X_train_scaled.head())

Data scaled robustly.


,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin,Pulse_Pressure,CK_MB_Troponin_Ratio
881,-0.222222,0.0,-0.500000,-0.558824,-0.421053,0.083333,2.362694,-0.102273,-0.458333,3.738599
723,-1.222222,-1.0,-0.681818,0.382353,0.473684,-0.222222,0.344560,-0.125000,0.166667,2.020633
889,0.388889,0.0,0.136364,-0.058824,-0.736842,-0.125000,-0.323834,5.340909,0.500000,-0.339300
1004,0.777778,-1.0,0.909091,-0.058824,-0.263158,3.847222,-0.448187,0.022727,0.125000,-0.228354
761,-0.166667,-1.0,-0.272727,-0.235294,0.105263,0.388889,-0.383420,0.909091,-0.416667,-0.320471


## 6. Export to Processed Data
The data preparation is complete. We will now export these numerical matrices to our folder structure so that our modeling notebooks can pick them up natively!

In [8]:
out_dir = os.path.join('data', 'processed', 'medical')
os.makedirs(out_dir, exist_ok=True)

X_train_scaled.to_csv(os.path.join(out_dir, 'X_train.csv'), index=False)
X_test_scaled.to_csv(os.path.join(out_dir, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(out_dir, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(out_dir, 'y_test.csv'), index=False)
print(f"Success! Exported 4 cleaned datasets to: {os.path.abspath(out_dir)}")

Success! Exported 4 cleaned datasets to: C:\Users\rahma\Desktop\machine learning project\Clustering_Medical data set\data\processed\medical
